# Scanpath locale

Notebook di lettura degli output scanpath del prompt sweep.


## Setup

Assunzioni:
- il notebook viene eseguito dal root della repository `MLLM-explainability`;
- `scripts/analysis/scanpath_viewer.py` e' presente;
- i risultati possono stare in locale oppure in una cartella condivisa esterna.


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import Image, display, Markdown

from scripts.analysis.scanpath_viewer import (
    load_json,
    build_scanpath_frames,
    save_gif,
    save_contact_sheet,
    safe_name,
)


## Selezione del metadata

Qui scegliamo il `metadata.json` da analizzare.


In [ ]:
PROJECT_ROOT = Path.cwd()
LOCAL_PROMPT_ROOT = PROJECT_ROOT / "outputs" / "prompt_sensitivity"
SHARED_RESULTS_ROOT = Path(r"G:\Drive condiviso\MLLM_explainability_results")  # aggiorna quando la cartella condivisa e' pronta
USE_SHARED_RESULTS = False
PREFERRED_PROMPT_LABEL = "00_baseline_neutral"
REQUIRE_SCANPATH = True

search_root = SHARED_RESULTS_ROOT if USE_SHARED_RESULTS else LOCAL_PROMPT_ROOT
if not search_root.exists():
    raise FileNotFoundError(f"Cartella risultati non trovata: {search_root}")

all_metadata = sorted(search_root.rglob("metadata.json"), key=lambda p: p.stat().st_mtime, reverse=True)
if not all_metadata:
    raise FileNotFoundError(f"Nessun metadata.json trovato in {search_root}")

scanpath_ready = []
for candidate in all_metadata:
    candidate_data = load_json(candidate)
    has_scanpath = isinstance(candidate_data.get("scanpath"), dict)
    if REQUIRE_SCANPATH and not has_scanpath:
        continue
    scanpath_ready.append((candidate, candidate_data))

if not scanpath_ready:
    raise RuntimeError("Nessun metadata con campi scanpath completi trovato.")

selected_path = None
selected_metadata = None
for candidate, candidate_data in scanpath_ready:
    if PREFERRED_PROMPT_LABEL and candidate.parent.name == PREFERRED_PROMPT_LABEL:
        selected_path = candidate
        selected_metadata = candidate_data
        break

if selected_path is None:
    selected_path, selected_metadata = scanpath_ready[0]

metadata_path = selected_path.resolve()
metadata = selected_metadata

display(Markdown(f"**Metadata selezionato:** `{metadata_path}`"))
display({
    "prompt_dir": metadata_path.parent.name,
    "run_dir": metadata_path.parent.parent.name,
    "step_records": len(metadata.get("step_records", [])),
    "has_scanpath": isinstance(metadata.get("scanpath"), dict),
})


## Parametri


In [ ]:
MAX_STEPS = None              # es: 120 per limitare il render
GIF_MS = 500                  # durata dei frame della GIF in ms
SHEET_COLS = 8                # colonne della contact sheet
SHOW_SECONDARY_TRACKS = False # True = mostra anche le tracce secondarie
DOMINANT_TAIL = 24            # parametro lasciato per compatibilita'

out_dir = metadata_path.parent / "scanpath_views"
out_dir.mkdir(parents=True, exist_ok=True)

scan_cfg = metadata.get("scanpath", {})
display(Markdown("### Configurazione scanpath salvata nel metadata"))
display(scan_cfg if scan_cfg else {"note": "scanpath non presente in questo metadata"})

def step_has_heatmap(step: dict) -> bool:
    heatmap_path = step.get("heatmap_path")
    return bool(heatmap_path) and Path(str(heatmap_path)).exists()

def frame_output_path(step: dict, output_dir: Path) -> Path:
    step_idx = int(step.get("step_idx", 0))
    token = safe_name(step.get("token_label", "tok"))
    return output_dir / f"step_{step_idx:04d}_{token}.png"


## Build dei frame


In [ ]:
frames = build_scanpath_frames(
    metadata,
    out_dir,
    max_steps=MAX_STEPS,
    show_secondary_tracks=SHOW_SECONDARY_TRACKS,
    dominant_tail=DOMINANT_TAIL,
)
gif_path = out_dir / "scanpath.gif"
sheet_path = out_dir / "scanpath_contact_sheet.jpg"

if SHEET_COLS < 1:
    raise ValueError("SHEET_COLS deve essere >= 1")

if not frames:
    raise RuntimeError("Nessun frame disponibile: controlla metadata e heatmap_path su disco.")

save_gif(frames, gif_path, duration_ms=GIF_MS)
save_contact_sheet(frames, sheet_path, cols=SHEET_COLS)

step_records = metadata.get("step_records", [])
if MAX_STEPS is not None:
    step_records = step_records[: max(0, int(MAX_STEPS))]

frame_entries = []
missing_steps = []
for step in step_records:
    out_path = frame_output_path(step, out_dir)
    if out_path.exists():
        frame_entries.append({"frame_path": out_path, "step": step})
    else:
        missing_steps.append({
            "step_idx": int(step.get("step_idx", -1)),
            "token_label": str(step.get("token_label", "")),
            "heatmap_exists": step_has_heatmap(step),
            "expected_frame": str(out_path),
        })

print(f"Frame generati: {len(frames)}")
print(f"Frame associati agli step: {len(frame_entries)}")
print(f"Step senza frame: {len(missing_steps)}")
print(f"GIF: {gif_path}")
print(f"Contact sheet: {sheet_path}")

if missing_steps:
    display(Markdown("### Step senza frame esportato"))
    display(pd.DataFrame(missing_steps).head(20))


## Anteprima rapida


In [ ]:
if gif_path.exists():
    display(Image(filename=str(gif_path)))
else:
    display(Markdown(f"GIF non trovata: `{gif_path}`"))

if sheet_path.exists():
    display(Image(filename=str(sheet_path)))
else:
    display(Markdown(f"Contact sheet non trovata: `{sheet_path}`"))


## Riepilogo dei frame

Qui controlliamo la corrispondenza tra frame esportati e step del metadata.


In [ ]:
if not frame_entries:
    raise RuntimeError("Nessun frame disponibile per il riepilogo finale")

frame_summary = pd.DataFrame([
    {
        "frame_index": idx,
        "step_idx": int(entry["step"].get("step_idx", idx)),
        "token_label": str(entry["step"].get("token_label", "")),
        "hotspots": len(entry["step"].get("hotspots", []) or []),
        "has_dominant_hotspot": bool(entry["step"].get("dominant_hotspot")),
        "frame_path": str(entry["frame_path"]),
    }
    for idx, entry in enumerate(frame_entries)
])

display(Markdown("### Riepilogo dei frame esportati"))
display(frame_summary.head(20))
display(Markdown(f"Frame disponibili: **{len(frame_summary)}**"))
display(Markdown("Ogni frame mostra il token in overlay e solo l'ultimo spostamento del percorso dominante."))


## Tabelle scanpath

Qui leggiamo le tracce principali e la loro durata.


In [ ]:
tracks = metadata.get("scanpath", {}).get("tracks", [])
if not tracks:
    display(Markdown("Nessun `scanpath.tracks` trovato in questo metadata."))
else:
    df_tracks = pd.DataFrame([
        {
            "track_id": t.get("track_id"),
            "num_points": t.get("num_points"),
            "start_step": t.get("start_step"),
            "end_step": t.get("end_step"),
            "mean_strength": t.get("mean_strength"),
        }
        for t in tracks
    ]).sort_values(["num_points", "mean_strength"], ascending=False)
    display(df_tracks.head(20))
    display(Markdown(f"Track totali: **{len(df_tracks)}**"))


## Coordinate del percorso dominante

Vista utile per controlli rapidi e confronti tra prompt.


In [ ]:
dominant = metadata.get("scanpath", {}).get("dominant_scanpath", [])
if not dominant:
    display(Markdown("Nessun dominant_scanpath disponibile."))
else:
    df_dom = pd.DataFrame(dominant)
    display(df_dom.head(30))
    display(Markdown(f"Punti dominant scanpath: **{len(df_dom)}**"))


## Note pratiche

- Se i metadata vecchi non hanno `scanpath`, devi rieseguire il prompt sweep con il runner aggiornato.
- Per ridurre il rumore: alza `scanpath-threshold-percentile` e/o `scanpath-min-hotspot-area` nel runner.
- Per un tracking meno frammentato: aumenta `scanpath-max-link-distance-ratio`.
- Quando i risultati saranno sul drive condiviso, basta aggiornare `SHARED_RESULTS_ROOT` e mettere `USE_SHARED_RESULTS = True`.
